# BindCraft: Protein binder design — local

The same cells as the Colab notebook, for a GPU box you own or rent. Run them in
order, top to bottom.

Three Colab cells are gone because they have no job here: the installer (done
once by `local/install.sh`), the Drive mount, and the compatibility patches —
the local env pins the Python and JAX versions upstream wants, so there is
nothing to shim.

**The run does not live in this notebook.** Cell 6 hands it to `run.sh`, which
starts it in a `tmux` session on the box. Closing this tab or dropping your SSH
connection does not stop it, and the display cells pick it back up.

If cell 1 says the kernel is wrong: *Kernel > Change kernel > Python (BindCraft)*.

In [ ]:
#@title 1. Check installation
import json, os, pathlib, subprocess, sys

HERE  = pathlib.Path.cwd()
LOCAL = HERE if (HERE / "run.sh").exists() else HERE / "local"
REPO  = LOCAL.parent
BC_ROOT = pathlib.Path(os.environ.get("BC_ROOT", "/workspace"))
BC_DIR  = BC_ROOT / "bindcraft"
SETTINGS_DIR = LOCAL / "settings"
STATE = {"settings": None, "design_path": str(BC_ROOT / "designs"),
         "advanced": None, "filters": None, "plddt": None, "clashes": None}

print("kernel :", sys.executable)
ok = True
for label, p in (("run.sh", LOCAL / "run.sh"),
                 ("bindcraft", BC_DIR / "bindcraft.py"),
                 ("AF2 weights", BC_DIR / "params" / "params_model_5_ptm.npz")):
    hit = p.exists()
    ok &= hit
    print(f"{label:12s}: {'OK     ' if hit else 'MISSING'} {p}")

try:
    import pandas, ipywidgets
    print("packages    : OK")
except ModuleNotFoundError as e:
    ok = False
    print(f"packages    : MISSING ({e.name})")
    print("\n>>> This notebook is running in the wrong Python.")
    print(">>> Kernel > Change kernel > 'Python (BindCraft)'.")
    print(">>> If that kernel is not listed, re-run: bash local/install.sh")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU         :", gpu or "none detected - designs would be useless")

if not ok:
    print("\nFix the above before running the cells below.")

In [ ]:
#@title 2. Binder design settings
import ipywidgets as W
from IPython.display import display, clear_output

def _md(t):
    return W.HTML(f"<div style='max-width:760px;margin:2px 0 6px'>{t}</div>")

row = dict(style={"description_width": "190px"}, layout=W.Layout(width="700px"))

def _expand(v):
    return str(v).replace("{{REPO}}", str(REPO)).replace("{{BC_ROOT}}", str(BC_ROOT))

_files = sorted(p.name for p in SETTINGS_DIR.glob("*.json"))
w_file     = W.Dropdown(options=_files + ["<new campaign>"],
                        value=_files[0] if _files else "<new campaign>",
                        description="Settings file", **row)
w_path     = W.Text(description="design_path", **row)
w_name     = W.Text(description="binder_name", **row)
w_pdb      = W.Text(description="starting_pdb", **row)
w_chains   = W.Text(description="chains", placeholder="A,C", **row)
w_hotspots = W.Text(description="target_hotspot_residues", placeholder="A31,A34 or blank", **row)
w_lengths  = W.IntRangeSlider(min=15, max=150, step=1, description="lengths",
                              continuous_update=False, **row)
w_designs  = W.IntText(description="number_of_final_designs", **row)
w_warn     = W.HTML()
w_out2     = W.Output()

def _check(*_):
    lo, _ = w_lengths.value
    w_warn.value = ("<b style='color:#c0392b'>Under ~20 residues a binder cannot fold "
                    "into anything that passes the quality gate — BindCraft will reject "
                    "every trajectory and loop without accepting a design.</b>"
                    if lo < 20 else "")

def _load(*_):
    name = w_file.value
    if name == "<new campaign>":
        s = {"design_path": f"{BC_ROOT}/designs/mytarget/", "binder_name": "mytarget",
             "starting_pdb": "", "chains": "A", "target_hotspot_residues": "",
             "lengths": [25, 40], "number_of_final_designs": 10}
    else:
        s = json.load(open(SETTINGS_DIR / name))
    w_path.value     = _expand(s.get("design_path", ""))
    w_name.value     = s.get("binder_name", "")
    w_pdb.value      = _expand(s.get("starting_pdb", ""))
    w_chains.value   = s.get("chains", "A")
    w_hotspots.value = s.get("target_hotspot_residues") or ""
    lo, hi           = s.get("lengths", [25, 40])
    w_lengths.value  = (int(lo), int(hi))
    w_designs.value  = int(s.get("number_of_final_designs", 10))
    _check()

def _collect():
    lo, hi = w_lengths.value
    return {"design_path": w_path.value.strip().rstrip("/") + "/",
            "binder_name": w_name.value.strip(),
            "starting_pdb": w_pdb.value.strip(),
            "chains": w_chains.value.strip(),
            "target_hotspot_residues": w_hotspots.value.strip(),
            "lengths": [int(lo), int(hi)],
            "number_of_final_designs": int(w_designs.value)}

def _save(_=None):
    with w_out2:
        clear_output()
        s = _collect()
        name = w_file.value
        if name == "<new campaign>":
            name = (s["binder_name"] or "mytarget") + ".json"
        path = SETTINGS_DIR / name
        path.write_text(json.dumps(s, indent=4) + "\n")
        if name not in w_file.options:
            w_file.options = sorted(p.name for p in SETTINGS_DIR.glob("*.json")) + ["<new campaign>"]
        w_file.value = name
        STATE["settings"], STATE["design_path"] = str(path), s["design_path"]
        print("Saved", path)
        if not os.path.isfile(s["starting_pdb"]):
            print("\nWARNING: starting_pdb does not exist:", s["starting_pdb"])
            print("BindCraft re-reads it only at the END of a trajectory, so a wrong")
            print("path costs a full trajectory before it errors. run.sh checks it too.")
        if " " in s["starting_pdb"] or " " in s["design_path"]:
            print("\nWARNING: a path contains a space. PyRosetta cannot open those.")

b_save = W.Button(description="Save settings", icon="save", button_style="primary")
b_save.on_click(_save)
w_file.observe(_load, names="value")
w_lengths.observe(_check, names="value")
_load()

display(W.VBox([
    _md("Pick an existing campaign to resume it, or <code>&lt;new campaign&gt;</code> "
        "to start one. These are the files in <code>local/settings/</code> — the same "
        "ones <code>run.sh</code> reads, so the notebook and the command line never "
        "disagree."),
    w_file,
    _md("Where designs are written. Give every campaign its own folder: BindCraft "
        "resumes from the CSVs it finds there, so leftovers from an old run get "
        "ranked alongside the new ones."),
    w_path, w_name,
    _md("The target <code>.pdb</code>, experimental or AlphaFold2. Trim it to as "
        "small as you need — the whole selected chain is backpropagated through the "
        "network and a big one costs real time. No waters, no ligands, no alternate "
        "conformations."),
    w_pdb,
    _md("Which chains to target, comma-separated. Other chains are ignored."),
    w_chains,
    _md("Which positions to aim at: <code>1,2-10</code>, chain-specific "
        "<code>A1-10,B1-20</code>, or whole chains <code>A</code>. Blank lets the "
        "pipeline pick a site."),
    w_hotspots,
    _md("Minimum and maximum binder size; sizes are sampled between them."),
    w_lengths,
    _md("How many designs passing the filters you want."),
    w_designs,
    w_warn, b_save, w_out2,
]))

In [ ]:
#@title 3. Advanced settings
w_design_proto = W.Dropdown(options=["Default", "Beta-sheet", "Peptide"],
                            value="Peptide", description="design_protocol", **row)
w_pred_proto   = W.Dropdown(options=["Default", "HardTarget"],
                            description="prediction_protocol", **row)
w_iface_proto  = W.Dropdown(options=["AlphaFold2", "MPNN"],
                            description="interface_protocol", **row)
w_tmpl_proto   = W.Dropdown(options=["Default", "Masked"],
                            description="template_protocol", **row)
w_adv_out      = W.Output()

_BASE = {"Default": "default_4stage_multimer",
         "Beta-sheet": "betasheet_4stage_multimer",
         "Peptide": "peptide_3stage_multimer"}

def _advanced_path():
    tag = _BASE[w_design_proto.value]
    if w_iface_proto.value == "MPNN":
        tag += "_mpnn"
    if w_tmpl_proto.value == "Masked":
        tag += "_flexible"
    if w_pred_proto.value == "HardTarget":
        tag += "_hardtarget"
    return BC_DIR / "settings_advanced" / f"{tag}.json"

def _show_adv(*_):
    with w_adv_out:
        clear_output()
        p = _advanced_path()
        if p.exists():
            STATE["advanced"] = str(p)
            print("advanced settings:", p.name)
        else:
            STATE["advanced"] = None
            print("No preset for that combination:", p.name)
            print("\nUpstream does not ship every combination. Available for this protocol:")
            for q in sorted((BC_DIR / "settings_advanced").glob(f"{_BASE[w_design_proto.value]}*.json")):
                print("  ", q.name)

for w in (w_design_proto, w_pred_proto, w_iface_proto, w_tmpl_proto):
    w.observe(_show_adv, names="value")
_show_adv()

display(W.VBox([
    _md("Which design protocol. <b>Default</b> is recommended. <b>Beta-sheet</b> "
        "promotes beta-sheeted binders but needs more sampling. <b>Peptide</b> is "
        "optimised for helical peptide binders."),
    w_design_proto,
    _md("Prediction protocol. <b>Default</b> predicts the binder from single "
        "sequence. <b>HardTarget</b> uses an initial guess to improve complex "
        "prediction on difficult targets, at the cost of some bias."),
    w_pred_proto,
    _md("Interface design. <b>AlphaFold2</b> generates the interface; <b>MPNN</b> "
        "optimises it with soluble MPNN."),
    w_iface_proto,
    _md("Target template. <b>Default</b> allows limited flexibility, <b>Masked</b> "
        "allows more on both sidechain and backbone."),
    w_tmpl_proto,
    w_adv_out,
]))

In [ ]:
#@title 4. Filters
_FILTERS = {"Default": "default_filters.json",
            "Peptide": "peptide_filters.json",
            "Relaxed": "relaxed_filters.json",
            "Peptide_Relaxed": "peptide_relaxed_filters.json",
            "None": "no_filters.json"}

w_filter  = W.Dropdown(options=list(_FILTERS), value="Peptide",
                       description="filter_option", **row)
w_filt_out = W.Output()

def _show_filt(*_):
    with w_filt_out:
        clear_output()
        p = BC_DIR / "settings_filters" / _FILTERS[w_filter.value]
        STATE["filters"] = str(p)
        print("filters:", p.name, "" if p.exists() else "  (MISSING)")

w_filter.observe(_show_filt, names="value")
_show_filt()

display(W.VBox([
    _md("<b>Default</b> is recommended. <b>Peptide</b> is for peptide binders. "
        "<b>Relaxed</b> is more permissive but may give fewer experimental "
        "successes. <b>Peptide_Relaxed</b> is for non-helical peptides. "
        "<b>None</b> is for benchmarking."),
    w_filter, w_filt_out,
]))

In [ ]:
#@title 5. Tuning controls
w_preset = W.Dropdown(options=["Production defaults", "Fast debug",
                               "Custom - use fields below"],
                      description="preset", **row)

w_soft    = W.IntText(value=75, description="soft_iterations", **row)
w_temp    = W.IntText(value=45, description="temporary_iterations", **row)
w_hard    = W.IntText(value=5,  description="hard_iterations", **row)
w_greedy  = W.IntText(value=15, description="greedy_iterations", **row)
w_numseq  = W.IntText(value=10, description="num_seqs", **row)
w_maxmpnn = W.IntText(value=2,  description="max_mpnn_sequences", **row)
w_recdes  = W.IntText(value=1,  description="num_recycles_design", **row)
w_recval  = W.IntText(value=3,  description="num_recycles_validation", **row)
w_sample  = W.Checkbox(value=True, description="sample_models", indent=True, **row)
w_maxtraj = W.IntText(value=0,  description="max_trajectories", **row)
w_anim    = W.Checkbox(value=True, description="save_design_animations", indent=True, **row)
w_plots   = W.Checkbox(value=True, description="save_design_trajectory_plots", indent=True, **row)
w_gate    = W.FloatText(value=0.70, step=0.05, description="trajectory_plddt_threshold", **row)
w_clash   = W.IntText(value=0, description="max_ca_clashes", **row)

_CUSTOM = [w_soft, w_temp, w_hard, w_greedy, w_numseq, w_maxmpnn, w_recdes,
           w_recval, w_sample, w_maxtraj, w_anim, w_plots, w_gate, w_clash]
w_tune_out = W.Output()

def _sync_tune(*_):
    custom = w_preset.value.startswith("Custom")
    for w in _CUSTOM:
        w.disabled = not custom
    with w_tune_out:
        clear_output()
        if w_preset.value == "Production defaults":
            print("Using this protocol's own settings_advanced JSON unchanged,")
            print("with the stock gates (0.70 final pLDDT, 0 CA clashes).")
        elif w_preset.value == "Fast debug":
            print("Iterations cut ~10x, gates dropped to 0.10 pLDDT / 100 clashes,")
            print("filters off, one design, animations and plots off.")
            print("DESIGNS FROM THIS MODE ARE STRUCTURALLY MEANINGLESS.")
        else:
            print("Using the fields above. They start at the Peptide protocol's")
            print("values; Default and Beta-sheet use num_seqs 20, so prefer")
            print("'Production defaults' over retyping a whole protocol by hand.")

w_preset.observe(_sync_tune, names="value")
_sync_tune()

display(W.VBox([
    _md("<b>Preset.</b> <code>Production defaults</code> is the reset — this "
        "protocol's own values, gates untouched. <code>Fast debug</code> exercises "
        "the pipeline end to end in minutes and produces meaningless designs. "
        "<code>Custom</code> uses the fields below."),
    w_preset, w_tune_out,
    _md("<b>Trajectory optimisation</b> — the biggest lever on runtime."),
    w_soft, w_temp, w_hard, w_greedy,
    _md("<b>MPNN sequence design</b>"), w_numseq, w_maxmpnn,
    _md("<b>AlphaFold2</b>"), w_recdes, w_recval, w_sample,
    _md("<b>Limits and output.</b> <code>max_trajectories</code> 0 means unlimited. "
        "It counts rows in <code>trajectory_stats.csv</code>, and rejected "
        "trajectories never write one, so it is no safeguard against a run that "
        "rejects everything."),
    w_maxtraj, w_anim, w_plots,
    _md("<b>Acceptance gates</b>, which upstream hardcodes in "
        "<code>colabdesign_utils.py</code> rather than exposing in the JSON. A "
        "trajectory is rejected below the pLDDT threshold or above the CA-clash "
        "count. Lowering either lets unfolded junk reach MPNN — use it to exercise "
        "the pipeline, never for real designs."),
    w_gate, w_clash,
]))

In [ ]:
#@title 6. Run BindCraft!
_FAST = dict(soft_iterations=8, temporary_iterations=5, hard_iterations=2,
             greedy_iterations=2, num_seqs=4, max_mpnn_sequences=1,
             save_design_animations=False, save_design_trajectory_plots=False)

w_run_out = W.Output()
b_go   = W.Button(description="Run BindCraft", button_style="success", icon="play")
b_stop = W.Button(description="Stop run", button_style="danger", icon="stop")

def _build_advanced():
    """Write a tuned copy of the chosen protocol JSON, or use it unchanged."""
    base = pathlib.Path(STATE["advanced"])
    if w_preset.value == "Production defaults":
        return base, None, None
    a = json.load(open(base))
    if w_preset.value == "Fast debug":
        a.update(_FAST)
        out = BC_ROOT / "advanced_notebook.json"
        out.write_text(json.dumps(a, indent=4) + "\n")
        return out, 0.10, 100
    a.update(soft_iterations=int(w_soft.value), temporary_iterations=int(w_temp.value),
             hard_iterations=int(w_hard.value), greedy_iterations=int(w_greedy.value),
             num_seqs=int(w_numseq.value), max_mpnn_sequences=int(w_maxmpnn.value),
             num_recycles_design=int(w_recdes.value),
             num_recycles_validation=int(w_recval.value),
             sample_models=bool(w_sample.value),
             max_trajectories=(int(w_maxtraj.value) or False),
             save_design_animations=bool(w_anim.value),
             save_design_trajectory_plots=bool(w_plots.value))
    out = BC_ROOT / "advanced_notebook.json"
    out.write_text(json.dumps(a, indent=4) + "\n")
    gate = float(w_gate.value) if abs(float(w_gate.value) - 0.70) > 1e-9 else None
    clash = int(w_clash.value) or None
    return out, gate, clash

def _running():
    return subprocess.run(["tmux", "has-session", "-t", "bindcraft"],
                          capture_output=True).returncode == 0

def _go(_):
    with w_run_out:
        clear_output()
        if _running():
            print("A run is already going in tmux session 'bindcraft'.")
            print("Stop it first, or you will lose it.")
            return
        if not STATE["advanced"]:
            print("Cell 3 has no valid protocol preset selected.")
            return
        _save()                      # cell 2: write the settings file first
        adv, gate, clash = _build_advanced()
        cmd = ["bash", str(LOCAL / "run.sh"), "-s", STATE["settings"],
               "-f", STATE["filters"], "-a", str(adv)]
        if w_preset.value == "Fast debug":
            cmd.append("--smoke")
        if gate is not None:
            cmd += ["--plddt", f"{gate:.2f}"]
        if clash is not None:
            cmd += ["--clashes", str(clash)]
        print(" ".join(cmd), "\n")
        r = subprocess.run(cmd, capture_output=True, text=True)
        print(r.stdout, r.stderr, sep="")
        if r.returncode:
            print(f"\nrun.sh exited {r.returncode} — nothing was launched.")
        else:
            print("\nRunning in tmux. Closing this tab will not stop it.")
            print("Use cell 7 to watch it.")

def _stop(_):
    with w_run_out:
        clear_output()
        if not _running():
            print("No run in progress.")
            return
        subprocess.run(["tmux", "kill-session", "-t", "bindcraft"])
        print("Stopped. Results already written to the design folder are kept,")
        print("and re-running resumes from them.")

b_go.on_click(_go)
b_stop.on_click(_stop)

display(W.VBox([
    _md("This writes the settings from the cells above and hands the campaign to "
        "<code>run.sh</code>, which starts it in a <code>tmux</code> session. The "
        "run belongs to the machine, not to this notebook — the browser can go "
        "away. A real campaign takes hours."),
    W.HBox([b_go, b_stop]), w_run_out,
]))

In [ ]:
#@title 7. Progress
import pandas as pd

w_prog = W.Output()
b_ref  = W.Button(description="Refresh", icon="refresh")

def _refresh(_=None):
    with w_prog:
        clear_output()
        dp = pathlib.Path(STATE["design_path"])
        print("tmux session 'bindcraft':", "running" if _running() else "not running")
        print("design folder           :", dp)
        if not dp.is_dir():
            print("\nNothing written yet.")
            return
        for label, fn in (("Sampled trajectories", "trajectory_stats.csv"),
                          ("MPNN designs", "mpnn_design_stats.csv"),
                          ("Accepted designs", "final_design_stats.csv")):
            f = dp / fn
            n = len(pd.read_csv(f)) if f.exists() else 0
            print(f"{label:22s} {n}")
        logs = sorted(dp.glob("run_*.log"))
        if logs:
            print(f"\n--- last 30 lines of {logs[-1].name} ---")
            print(subprocess.run(["tail", "-n", "30", str(logs[-1])],
                                 capture_output=True, text=True).stdout)

b_ref.on_click(_refresh)
display(W.VBox([_md("Press Refresh. Nothing here polls on its own, so this cell "
                    "never holds the kernel."), b_ref, w_prog]))
_refresh()

In [ ]:
#@title 8. Top 20 Designs
import pandas as pd

dp = pathlib.Path(STATE["design_path"])
final = dp / "final_design_stats.csv"

if not final.exists():
    print("No final_design_stats.csv yet — nothing has been ranked.")
else:
    df = pd.read_csv(final)
    if df.empty:
        print("No designs accepted yet. Cell 11 says why trajectories are failing.")
    else:
        cols = [c for c in ["Rank", "Design", "Sequence", "Length", "pLDDT", "i_pTM",
                            "pAE", "i_pAE", "Binder_Energy_Score",
                            "ShapeComplementarity", "Interface_Hydrophobicity",
                            "n_InterfaceResidues", "Hotspot_RMSD"] if c in df.columns]
        out = df[cols] if cols else df
        if "Rank" in out.columns:
            out = out.sort_values("Rank")
        pd.set_option("display.max_colwidth", 60)
        display(out.head(20))
        print(f"\n{len(df)} accepted design(s) total.")

In [ ]:
#@title 9. Top Design Display
import pathlib

dp = pathlib.Path(STATE["design_path"])
ranked = sorted((dp / "Accepted" / "Ranked").glob("*.pdb"))

if not ranked:
    print("Nothing in Accepted/Ranked yet.")
else:
    pdb = ranked[0]
    print(f"{len(ranked)} ranked design(s). Showing {pdb.name}")
    try:
        import py3Dmol
        v = py3Dmol.view(width=780, height=520)
        v.addModel(pdb.read_text(), "pdb")
        v.setStyle({"cartoon": {"color": "lightgrey"}})
        v.setStyle({"chain": "B"}, {"cartoon": {"color": "orange"}})
        v.zoomTo()
        v.show()
        print("Target grey, binder orange.")
    except Exception as e:
        print("Could not render:", e)

In [ ]:
#@title 10. Display animation
import pathlib
from IPython.display import HTML, display

dp = pathlib.Path(STATE["design_path"])
anims = sorted((dp / "Accepted" / "Animation").glob("*.html"))

if not anims:
    print("No animations. They are off in Fast debug and in any run with")
    print("save_design_animations unchecked.")
else:
    print(f"{len(anims)} animation(s). Showing {anims[0].name}")
    # Pass the file's CONTENTS, not its path - HTML(path) renders the path as text.
    display(HTML(anims[0].read_text()))

In [ ]:
#@title 11. Diagnose rejected trajectories
import os, pathlib
from itertools import combinations
import pandas as pd

max_pdbs = 6

dp = pathlib.Path(STATE["design_path"])

# ---- 0. will the next run do anything at all? -----------------------------
# The loop counts .pdb files in Accepted/ before sampling and breaks if that
# count has reached number_of_final_designs, printing a message that names the
# count it found rather than your target.
acc = dp / "Accepted"
have = len([f for f in acc.glob("*.pdb") if not f.name.startswith(".")]) if acc.is_dir() else 0
want = None
if STATE["settings"] and os.path.isfile(STATE["settings"]):
    want = int(json.load(open(STATE["settings"]))["number_of_final_designs"])

print(f"Accepted designs on disk: {have}" + (f"   target: {want}" if want else ""))
if want and have >= want:
    print("   -> the next run reranks and exits in seconds without designing")
    print(f"      anything. Raise number_of_final_designs above {have} in cell 2,")
    print("      save, then run again.")
elif want:
    print(f"   -> {want - have} more to find")
print()

# ---- 1. the tally BindCraft already keeps ---------------------------------
fail = dp / "failure_csv.csv"
if not fail.exists():
    cand = list(dp.glob("*failure*.csv"))
    fail = cand[0] if cand else fail
if fail.exists():
    f = pd.read_csv(fail)
    tally = {c: int(f[c].sum()) for c in f.columns if f[c].dtype != object}
    tally = {k: v for k, v in sorted(tally.items(), key=lambda kv: -kv[1]) if v}
    if tally:
        print("Why designs were rejected:")
        for k, v in tally.items():
            print(f"  {v:6d}  {k}")
    else:
        print("Failure tally is empty.")
else:
    print("No failure CSV found in", dp)
print()

# ---- 2. where the clashes are --------------------------------------------
# The gate counts any two CA atoms closer than 2.5 A (same residue and directly
# sequential pairs excluded). Because it looks at CA only it does not skip
# same-chain pairs, so a collapsed binder trips it just as readily as a binder
# driven into the target. This says which.
def _ca(path):
    out = []
    for line in open(path):
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            out.append((line[21], int(line[22:26]),
                        (float(line[30:38]), float(line[38:46]), float(line[46:54]))))
    return out

pdbs = sorted((dp / "Trajectory").glob("*.pdb"))[:max_pdbs]
if not pdbs:
    print("No trajectory PDBs in", dp / "Trajectory")
else:
    for p in pdbs:
        atoms = _ca(p)
        same = cross = 0
        for (c1, r1, x1), (c2, r2, x2) in combinations(atoms, 2):
            if c1 == c2 and abs(r1 - r2) <= 1:
                continue
            d2 = sum((a - b) ** 2 for a, b in zip(x1, x2))
            if d2 < 2.5 ** 2:
                if c1 == c2:
                    same += 1
                else:
                    cross += 1
        verdict = ("clean" if same + cross == 0 else
                   "binder collapsed on itself" if cross == 0 else
                   "binder into target" if same == 0 else "both")
        print(f"{p.name:48s} same-chain {same:4d}  cross-chain {cross:4d}   {verdict}")
    print("\nSame-chain clashes mean the binder folded into itself: a length or")
    print("protocol problem. Cross-chain means it was driven into the target:")
    print("usually the hotspots.")